# 00 · Preparación de los datos

**Proyecto:** Reproducción y extensión de *MicrofluidicML* (Moore et al., *Adv. Funct. Mater.* 2026, e14387).

**Objetivo de este notebook:** transformar la base de datos cruda del autor (`recursos/data/mfNP_database.csv`) en las tablas que reciben los modelos, **replicando exactamente** la manipulación que hace el código R del autor:

| Paso | Función del autor | Dónde |
|---|---|---|
| 1 | `prep_main()`: limpieza general (0 → `NA`) | `recursos/extra/functions.R`, líneas 83–158 |
| 2 | Resumen por formulación → `df_summary.csv` | `recursos/01_mfNP.R`, líneas 80–139 |
| 3 | Separación lípidos / PLGA | `recursos/01_mfNP.R`, líneas 142–143 |
| 4 | `ml_prep()`: tabla para ML (NA → 0, razón material:fármaco, descriptores del fármaco, una columna por reactivo) | `recursos/extra/functions.R`, líneas 238–388 |

Además se añade un **identificador de formulación** (`meta_formulation_id`), que el código del autor descarta. **No es un predictor.** Se usa en la Parte C para medir la fuga por réplicas y en la Parte D para la partición agrupada.

**Salidas** (las leen los notebooks de las Partes B y C):
- `datos_procesados/ml_lipidos.csv`: equivalente a `dfML_lipid` del autor + columnas `meta_*`.
- `datos_procesados/ml_plga.csv`: equivalente a `dfML_PNP` del autor + columnas `meta_*`.

> La explicación detallada de cada transformación y sus consecuencias está en `Analisis_datos_y_codigo_R.md` (secciones 4–6). Los supuestos del proyecto están en `docs/supuestos_y_validaciones.md`.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Raíz del proyecto (funciona si el notebook se ejecuta desde notebooks/ o desde la raíz)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUTA_CRUDA = ROOT / "recursos" / "data" / "mfNP_database.csv"
RUTA_RESUMEN_AUTOR = ROOT / "recursos" / "data" / "df_summary.csv"
SALIDA = ROOT / "datos_procesados"
SALIDA.mkdir(exist_ok=True)

# Constante del autor (01_mfNP.R): umbral de polidispersidad
PDI_LIM = 0.250

## 1. Datos crudos

Cada fila es **una medición** (una lectura de DLS / UV-Vis) de una nanopartícula fabricada en el chip microfluídico. Cada formulación ("receta") se fabricó y midió **3 veces**.

In [ ]:
df_crudo = pd.read_csv(RUTA_CRUDA)
print("Dimensiones:", df_crudo.shape)
df_crudo.head(3)

In [ ]:
# Composición de la base por tipo de partícula y presencia de fármaco
resumen = (df_crudo.assign(payload=df_crudo["payload"].fillna("sin fármaco"))
           .groupby(["class", "np_type", "payload"]).size().rename("mediciones").to_frame())
resumen["formulaciones (÷3)"] = resumen["mediciones"] // 3
resumen

In [ ]:
# Espacio de diseño: niveles de cada variable de proceso por tipo de partícula
diseno = df_crudo.groupby("np_type").agg(
    TFR=("tfr", lambda s: sorted(s.unique().tolist())),
    FRR=("frr", lambda s: sorted(s.unique().tolist())),
    conc_total_mM=("totOrg_conc", lambda s: sorted(np.round(s.unique(), 3).tolist())),
)
diseno

## 2. Paso 1: `prep_main()`, limpieza general

El autor:
1. convierte 19 columnas a numéricas (ya lo son);
2. unifica nombres de `np_type` (ya vienen unificados);
3. **reemplaza todos los `0` y los textos vacíos por `NA`** (`x[x == 0] <- NA`);
4. crea columnas descriptivas de texto (`orgReagents`, `orgRatios`, …) y un `id` de formulación que **no llegan al ML**. Aquí se omiten porque no afectan a ninguna salida.

In [ ]:
def prep_main(x: pd.DataFrame) -> pd.DataFrame:
    # Emula prep_main() de recursos/extra/functions.R (solo los pasos que afectan al ML)
    x = x.copy()
    # Unificación de nombres de np_type (sin efecto en estos datos; se conserva por fidelidad)
    x["np_type"] = x["np_type"].replace({
        "Liposomes": "Liposome",
        "Lipid Nanoparticles": "Lipid Nanoparticle",
        "Polymeric Nanoparticles": "Polymeric Nanoparticle",
        "Solid-lipid Nanoparticles": "Solid-lipid Nanoparticle",
    })
    # x[x == 0] <- NA ; x[x == ""] <- NA ; x[x == "NA"] <- NA
    num = x.select_dtypes("number").columns
    x[num] = x[num].mask(x[num] == 0)
    txt = x.columns.difference(num)
    x[txt] = x[txt].mask(x[txt].isin(["", "NA"]))
    return x


df = prep_main(df_crudo)

# Cuántos ceros se volvieron NA en cada columna
ceros = (df_crudo.select_dtypes("number") == 0).sum()
ceros[ceros > 0].rename("ceros → NA")

In [ ]:
# 01_mfNP.R: etiqueta de dispersidad por medición
df["dispersity"] = np.where(df["pdi"] <= PDI_LIM, "monodisperse", "polydisperse")
df["dispersity"].value_counts()

## 3. Paso 2: resumen por formulación (`df_summary.csv`)

El autor agrupa por todas las columnas de la receta y calcula la media y la desviación estándar de dH, PdI y EE. **Este resumen no se usa para entrenar modelos**, pero sirve como **prueba de fidelidad**: si nuestra emulación de `prep_main` es correcta, debemos obtener las mismas 189 filas y los mismos promedios que el archivo `df_summary.csv` que entregó el autor.

In [ ]:
claves = ["class", "np_type", "tfr", "frr", "org_phase", "aq_phase", "totOrg_conc", "totAq_conc",
          "orgReag1_reag", "orgReag1_conc", "orgReag2_reag", "orgReag2_conc",
          "orgReag3_reag", "orgReag3_conc", "orgReag4_reag", "orgReag4_conc",
          "orgReag5_reag", "orgReag5_conc", "orgReag6_reag", "orgReag6_conc",
          "aqReag1_reag", "aqReag1_conc", "aqReag2_reag", "aqReag2_conc",
          "payload", "payloadConc_mM"]

# R calcula mean() sin na.rm: si hay un NA en el grupo, la media es NA (skipna=False)
resumen_py = (df.groupby(claves, dropna=False)
                .agg(meanDiam=("hdiam_nm", "mean"),
                     meanPdI=("pdi", "mean"),
                     meanEE=("percEE", lambda s: s.mean(skipna=False)))
                .reset_index())
resumen_autor = pd.read_csv(RUTA_RESUMEN_AUTOR)

print("Formulaciones (nuestra emulación):", len(resumen_py))
print("Formulaciones (df_summary.csv del autor):", len(resumen_autor))
assert len(resumen_py) == len(resumen_autor) == 189

# Comparación de promedios (ordenados, para que el orden de filas no importe)
for col in ["meanDiam", "meanPdI"]:
    a = np.sort(resumen_py[col].to_numpy())
    b = np.sort(resumen_autor[col].to_numpy())
    assert np.allclose(a, b), col
assert resumen_py["meanEE"].isna().sum() == resumen_autor["meanEE"].isna().sum() == 81
print("✔ Los promedios de dH y PdI coinciden con df_summary.csv; la EE es NA en las mismas 81 formulaciones vacías.")

## 4. Paso 3: separación por familia

LNP y liposomas se modelan **juntos** como "lípidos"; PLGA por separado.

In [ ]:
df_lipidos = df[df["class"] == "Lipid-based Nanoparticles"].copy()
df_plga = df[df["class"] == "Polymeric Nanoparticles"].copy()
print("Lípidos:", df_lipidos.shape[0], "mediciones | PLGA:", df_plga.shape[0], "mediciones")

## 5. Paso 4: `ml_prep()`, la tabla para ML

| # | Acción del autor |
|---|---|
| 4.1 | `NA → 0` en todo el data frame (el "vacío" vuelve a ser 0) |
| 4.2 | `reag_payload_ratio = (totOrg_conc + totAq_conc) / payloadConc_mM`; la división por 0 (`Inf`) se reemplaza por 0 |
| 4.3 | Descriptores del fármaco (`payloadMW`, `XlogP3`, `polarSA`) desde la tabla `dat_payload` (PubChem) |
| 4.4 | Selección de columnas (se descartan `np_type`, solventes, `totOrg_conc`, `payloadConc_mM`, ids…) |
| 4.5 | Pivot "una columna por reactivo" con su concentración (0 si el reactivo no está) |
| 4.6 | Los reactivos acuosos (todos 0) desaparecen |
| 4.7 | `NA → 0` otra vez y limpieza de nombres (`-`, espacios y `:` → `.`; paréntesis → `_`) |

La tabla `dat_payload` se copia tal cual de `recursos/extra/functions.R` (11 fármacos; en estos datos solo aparece la curcumina).

In [ ]:
# dat_payload de recursos/extra/functions.R (columnas que usa ml_prep)
DAT_PAYLOAD = pd.DataFrame({
    "payload": ["5-Fluorouracil", "Curcumin", "Cyclophosphamide", "Daunorubicin", "Docetaxel",
                "Doxorubicin", "Epirubicin", "Methotrexate", "O6-benzylguanine", "Paclitaxel", "Temozolomide"],
    "MW": [130.08, 368.4, 261.08, 527.5, 807.9, 543.5, 543.5, 454.4, 241.25, 853.9, 194.15],
    "XlogP3": [-0.9, 3.2, 0.6, 1.8, 1.6, 1.3, 1.3, -1.8, 1.2, 2.5, -1.1],
    "polarSA_angSquared": [58.2, 93.1, 41.6, 186, 224, 206, 206, 211, 89.7, 221, 106],
}).set_index("payload")

OBJETIVOS = ["hdiam_nm", "pdi", "zetaPot_mV", "percEE"]


def limpiar_nombre(nombre: str) -> str:
    # Replica los gsub() del final de ml_prep()
    nombre = re.sub(r"\s+", ".", nombre)
    for a, b in [("-", "."), (":", "."), ("(", "_"), (")", "_")]:
        nombre = nombre.replace(a, b)
    return nombre


def ml_prep(x: pd.DataFrame) -> pd.DataFrame:
    # Emula ml_prep() de recursos/extra/functions.R
    x = x.copy().reset_index(drop=True)
    # 4.1  x[is.na(x)] <- 0  (en R, en las columnas de texto el NA pasa a ser el texto "0")
    num = x.select_dtypes("number").columns
    x[num] = x[num].fillna(0)
    txt = x.columns.difference(num)
    x[txt] = x[txt].fillna("0")
    # 4.2  razón material:fármaco; Inf (sin fármaco) -> 0
    ratio = (x["totOrg_conc"] + x["totAq_conc"]) / x["payloadConc_mM"]
    x["reag_payload_ratio"] = ratio.replace([np.inf, -np.inf], np.nan).fillna(0)
    # 4.3  descriptores del fármaco (sin fármaco -> NA -> 0 en el paso 4.7)
    x["payloadMW"] = x["payload"].map(DAT_PAYLOAD["MW"])
    x["XlogP3"] = x["payload"].map(DAT_PAYLOAD["XlogP3"])
    x["polarSA"] = x["payload"].map(DAT_PAYLOAD["polarSA_angSquared"])
    # 4.5  una columna por reactivo orgánico (suma de concentraciones por nombre)
    largo = pd.concat([
        pd.DataFrame({"fila": x.index, "reag": x[f"orgReag{i}_reag"].astype(str),
                      "conc": x[f"orgReag{i}_conc"]})
        for i in range(1, 7)
    ])
    VACIO = ["0", "0.0"]  # una columna de reactivos sin datos es numérica, y su 0 se lee como "0.0"
    largo = largo[~largo["reag"].isin(VACIO)]       # huecos vacíos (la columna "0" en R)
    reactivos = (largo.pivot_table(index="fila", columns="reag", values="conc", aggfunc="sum")
                      .reindex(x.index).fillna(0))
    reactivos = reactivos[sorted(reactivos.columns)]
    reactivos.columns.name = None
    # 4.6  reactivos acuosos: todos vacíos -> no generan columnas
    assert x[["aqReag1_reag", "aqReag2_reag"]].astype(str).isin(VACIO).all().all()
    # 4.4  selección de columnas + reensamblado (reactivos primero, como en R)
    resto = x[["tfr", "frr", "payload", "reag_payload_ratio", "payloadMW", "XlogP3", "polarSA"] + OBJETIVOS]
    salida = pd.concat([reactivos, resto], axis=1)
    # 4.7  NA -> 0 y limpieza de nombres
    salida = salida.fillna(0)
    salida.columns = [limpiar_nombre(c) for c in salida.columns]
    return salida


ml_lipidos = ml_prep(df_lipidos)
ml_plga = ml_prep(df_plga)
print("dfML_lipid:", ml_lipidos.shape, "| dfML_PNP:", ml_plga.shape)
ml_lipidos.head(3)

### Verificación de fidelidad

Según el análisis del código R (`Analisis_datos_y_codigo_R.md`, §5), las tablas del autor tienen **243 × 16** (lípidos) y **324 × 12** (PLGA), con estas columnas exactas.

In [ ]:
COLS_LIPIDOS = ["Cholesterol", "DODAP", "DPPC", "DSPC", "DSPE.PEG2k", "tfr", "frr", "payload",
                "reag_payload_ratio", "payloadMW", "XlogP3", "polarSA",
                "hdiam_nm", "pdi", "zetaPot_mV", "percEE"]
COLS_PLGA = ["Resomer.RG.504.H", "tfr", "frr", "payload", "reag_payload_ratio", "payloadMW",
             "XlogP3", "polarSA", "hdiam_nm", "pdi", "zetaPot_mV", "percEE"]

assert ml_lipidos.shape == (243, 16) and list(ml_lipidos.columns) == COLS_LIPIDOS
assert ml_plga.shape == (324, 12) and list(ml_plga.columns) == COLS_PLGA
print("✔ Dimensiones y columnas idénticas a dfML_lipid / dfML_PNP del autor.")

# Valores que toma la razón material:fármaco (0 = sin fármaco)
print("reag_payload_ratio lípidos:", np.round(np.sort(ml_lipidos["reag_payload_ratio"].unique()), 2))
print("reag_payload_ratio PLGA:   ", np.round(np.sort(ml_plga["reag_payload_ratio"].unique()), 3))

## 6. Identificador de formulación (no es predictor)

Las 3 réplicas de una formulación tienen **todas las variables de entrada idénticas**. Por eso el identificador se reconstruye como la combinación única de las X, redondeadas a 6 decimales para evitar diferencias de redondeo como `0.1357220412595` vs `0.135722041259501`.

También se guardan como metadatos el tipo de partícula y el lote (`commit_id`), útiles para la Parte D. Llevan el prefijo `meta_` y quedan **excluidos de los modelos**.

In [ ]:
def agregar_metadatos(ml: pd.DataFrame, original: pd.DataFrame, prefijo: str) -> pd.DataFrame:
    ml = ml.copy()
    predictores = [c for c in ml.columns if c not in OBJETIVOS + ["payload"]]
    clave = ml[predictores].round(6).astype(str).agg("|".join, axis=1)
    codigos, _ = pd.factorize(clave)
    ml["meta_formulation_id"] = [f"{prefijo}{c + 1:03d}" for c in codigos]
    ml["meta_np_type"] = original["np_type"].to_numpy()
    ml["meta_commit_id"] = original["commit_id"].to_numpy()
    return ml


ml_lipidos = agregar_metadatos(ml_lipidos, df_lipidos, "L")
ml_plga = agregar_metadatos(ml_plga, df_plga, "P")

for nombre, d, esperado in [("Lípidos", ml_lipidos, 81), ("PLGA", ml_plga, 108)]:
    n_form = d["meta_formulation_id"].nunique()
    reps = d.groupby("meta_formulation_id").size()
    print(f"{nombre}: {n_form} formulaciones; réplicas por formulación = {sorted(reps.unique())}")
    assert n_form == esperado and (reps == 3).all()
    # Cada formulación pertenece a un solo tipo de partícula
    assert (d.groupby("meta_formulation_id")["meta_np_type"].nunique() == 1).all()
print("✔ 189 formulaciones con exactamente 3 réplicas cada una.")

## 7. Problemas de calidad (se reportan, **no se corrigen**)

Para que la reproducción (Parte B) sea fiel, los datos se usan **tal como los usó el autor**. Aquí solo se documentan los problemas; su posible efecto se discute en `docs/supuestos_y_validaciones.md`.

In [ ]:
calidad = []
z = df_crudo["zetaPot_mV"]
calidad.append(("Potencial zeta imposible (|ζ| > 200 mV)", int((z.abs() > 200).sum()),
                f"valor: {z[z.abs() > 200].tolist()} mV (no es predictor ni objetivo)"))
ee = df_crudo["percEE"]
calidad.append(("EE > 100 %", int((ee > 100).sum()),
                f"máx = {ee.max():.1f} % (liposomas con curcumina); se usan tal cual"))
lnp = df_crudo[df_crudo["np_type"] == "Lipid Nanoparticle"]
calidad.append(("EE de LNP (nunca llevan fármaco)", len(lnp),
                f"{int(lnp['percEE'].isna().sum())} NA + {int((lnp['percEE'] == 0).sum())} ceros → todas pasan a EE = 0"))
cv_l = ml_lipidos.groupby("meta_formulation_id")["hdiam_nm"].agg(lambda s: s.std() / s.mean())
calidad.append(("Triplicados con CV de dH > 30 % (lípidos)", int((cv_l > 0.30).sum()),
                f"CV máx = {cv_l.max():.2f}"))
cv_p = ml_plga.groupby("meta_formulation_id")["hdiam_nm"].agg(lambda s: s.std() / s.mean())
calidad.append(("Triplicados con CV de dH > 30 % (PLGA)", int((cv_p > 0.30).sum()),
                f"CV máx = {cv_p.max():.2f}"))
pd.DataFrame(calidad, columns=["Problema", "Filas / formulaciones", "Detalle"])

In [ ]:
# Filtro PdI <= 0.25 de los modelos de diámetro: se aplica por medición y rompe triplicados
def triplicados_tras_filtro(d):
    n_ok = d.assign(ok=d["pdi"] <= PDI_LIM).groupby("meta_formulation_id")["ok"].sum()
    return n_ok.value_counts().reindex([3, 2, 1, 0], fill_value=0)


pd.DataFrame({"Lípidos": triplicados_tras_filtro(ml_lipidos),
              "PLGA": triplicados_tras_filtro(ml_plga)}).rename_axis("réplicas con PdI ≤ 0.25")

## 8. Guardar las tablas procesadas

In [ ]:
ml_lipidos.to_csv(SALIDA / "ml_lipidos.csv", index=False)
ml_plga.to_csv(SALIDA / "ml_plga.csv", index=False)

# Comprobación de lectura (ida y vuelta)
for nombre in ["ml_lipidos.csv", "ml_plga.csv"]:
    d = pd.read_csv(SALIDA / nombre)
    print(f"{nombre}: {d.shape} → datos_procesados/{nombre}")

## Resumen

- La emulación de `prep_main()` reproduce **exactamente** el `df_summary.csv` del autor: 189 formulaciones y los mismos promedios.
- La emulación de `ml_prep()` produce tablas con **las mismas dimensiones y columnas** que `dfML_lipid` (243 × 16) y `dfML_PNP` (324 × 12).
- Se añadió `meta_formulation_id` (81 + 108 = 189 formulaciones × 3 réplicas). El pipeline del autor no lo conserva, y es la base del gap (Parte C).
- Los problemas de calidad se documentan pero **no se corrigen**, para mantener la fidelidad con el autor.